In [1]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin

def scrape_chemxplore_articles(base_url="https://chemxplore.com/news", max_pages=None):
    """
    Scrape all article URLs from ChemXplore news pages
    
    Args:
        base_url: Base URL for the news section
        max_pages: Maximum number of pages to scrape (None for all available)
    
    Returns:
        List of article URLs
    """
    
    all_article_urls = []
    page_num = 1
    
    # Headers to mimic a real browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    
    print(f"�� Starting to scrape ChemXplore articles from: {base_url}")
    
    while True:
        # Construct the page URL
        if page_num == 1:
            page_url = base_url
        else:
            page_url = f"{base_url}?page={page_num}"
        
        print(f"\n📄 Scraping page {page_num}: {page_url}")
        
        try:
            # Make the request
            response = requests.get(page_url, headers=headers, timeout=30)
            response.raise_for_status()
            
            # Parse the HTML
            soup = BeautifulSoup(response.text, "html.parser")
            
            # Find the news item list
            news_list = soup.find("ul", class_="news-item-list")
            
            if not news_list:
                print(f"❌ No news list found on page {page_num}")
                break
            
            # Find all news item previews
            news_items = news_list.find_all("div", class_="news-item-preview")
            
            if not news_items:
                print(f"❌ No news items found on page {page_num}")
                break
            
            print(f"✅ Found {len(news_items)} articles on page {page_num}")
            
            # Extract URLs from each news item
            page_article_urls = []
            for item in news_items:
                link = item.find("a", class_="news-item-preview__main")
                if link and link.get("href"):
                    article_url = link["href"]
                    # Convert relative URLs to absolute URLs
                    if article_url.startswith("/"):
                        article_url = urljoin("https://chemxplore.com", article_url)
                    page_article_urls.append(article_url)
            
            # Add to the main list
            all_article_urls.extend(page_article_urls)
            print(f"📝 Added {len(page_article_urls)} article URLs from page {page_num}")
            
            # Check if we've reached the maximum pages
            if max_pages and page_num >= max_pages:
                print(f"✅ Reached maximum page limit ({max_pages})")
                break
            
            # Check if there are more pages (look for pagination or next button)
            # You might need to adjust this based on the actual pagination structure
            next_page_exists = check_next_page_exists(soup, page_num)
            
            if not next_page_exists:
                print(f"✅ No more pages found after page {page_num}")
                break
            
            page_num += 1
            
            # Add a small delay to be respectful to the server
            time.sleep(1)
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Error scraping page {page_num}: {e}")
            break
        except Exception as e:
            print(f"❌ Unexpected error on page {page_num}: {e}")
            break
    
    print(f"\n�� Scraping completed!")
    print(f"📊 Total articles found: {len(all_article_urls)}")
    print(f"📄 Total pages scraped: {page_num}")
    
    return all_article_urls

def check_next_page_exists(soup, current_page):
    """
    Check if there's a next page available
    You may need to adjust this based on the actual pagination structure
    """
    # Look for common pagination indicators
    # This is a basic check - you might need to customize based on the actual site
    
    # Check if there are more news items (if less than expected, might be last page)
    news_list = soup.find("ul", class_="news-item-list")
    if news_list:
        news_items = news_list.find_all("div", class_="news-item-preview")
        # If we get significantly fewer items than expected, it might be the last page
        if len(news_items) < 20:  # Adjust this number based on typical page size
            return False
    
    # Look for next page button or pagination
    next_buttons = soup.find_all("a", string=lambda text: text and "next" in text.lower())
    if next_buttons:
        return True
    
    # Look for page numbers that are higher than current
    page_links = soup.find_all("a", href=lambda href: href and "page=" in href)
    for link in page_links:
        href = link["href"]
        if "page=" in href:
            try:
                page_num = int(href.split("page=")[1])
                if page_num > current_page:
                    return True
            except ValueError:
                continue
    
    # If we can't determine, assume there might be more pages
    return True

def save_urls_to_file(urls, filename="chemxplore_article_urls.txt"):
    """Save the URLs to a text file"""
    with open(filename, "w", encoding="utf-8") as f:
        for url in urls:
            f.write(url + "\n")
    print(f"💾 URLs saved to: {filename}")

def main():
    """Main function to run the scraper"""
    
    # Scrape all available pages (set max_pages to limit if needed)
    article_urls = scrape_chemxplore_articles(max_pages=None)
    
    if article_urls:
        # Display first few URLs as examples
        print(f"\n📋 First 5 article URLs:")
        for i, url in enumerate(article_urls[:5], 1):
            print(f"{i}. {url}")
        
        if len(article_urls) > 5:
            print(f"... and {len(article_urls) - 5} more")
        
        # Save to file
        save_urls_to_file(article_urls)
        
        return article_urls
    else:
        print("❌ No articles found")
        return []

if __name__ == "__main__":
    # Run the scraper
    urls = main()

�� Starting to scrape ChemXplore articles from: https://chemxplore.com/news

📄 Scraping page 1: https://chemxplore.com/news


/Users/mac/Documents/Tuone/tuone_v6/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Found 6 articles on page 1
📝 Added 6 article URLs from page 1
✅ No more pages found after page 1

�� Scraping completed!
📊 Total articles found: 6
📄 Total pages scraped: 1

📋 First 5 article URLs:
1. https://chemxplore.com/news/lanxess-takes-action-to-counter-weak-market-environment
2. https://chemxplore.com/news/porthos-co2-pipeline-laid-on-seabed-and-entrenched
3. https://chemxplore.com/news/ercros-reduces-its-co2-emissions-by-38-in-4-years
4. https://chemxplore.com/news/lyten-to-acquire-all-remaining-northvolt-assets-in-sweden-and-germany
5. https://chemxplore.com/news/clariant-advances-plastics-stabilizer-technology-with-expanded-production-and-new-applications
... and 1 more
💾 URLs saved to: chemxplore_article_urls.txt


In [2]:
# Scrape all available pages
urls = scrape_chemxplore_articles()

# Or limit to specific number of pages
urls = scrape_chemxplore_articles(max_pages=5)

# The function returns a list of all article URLs
print(f"Found {len(urls)} articles")

�� Starting to scrape ChemXplore articles from: https://chemxplore.com/news

📄 Scraping page 1: https://chemxplore.com/news
✅ Found 6 articles on page 1
📝 Added 6 article URLs from page 1
✅ No more pages found after page 1

�� Scraping completed!
📊 Total articles found: 6
📄 Total pages scraped: 1
�� Starting to scrape ChemXplore articles from: https://chemxplore.com/news

📄 Scraping page 1: https://chemxplore.com/news
✅ Found 6 articles on page 1
📝 Added 6 article URLs from page 1
✅ No more pages found after page 1

�� Scraping completed!
📊 Total articles found: 6
📄 Total pages scraped: 1
Found 6 articles


In [3]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin

def scrape_chemxplore_articles(base_url="https://chemxplore.com/news", max_pages=None):
    """
    Scrape all article URLs from ChemXplore news pages

    Args:
        base_url: Base URL for the news section
        max_pages: Maximum number of pages to scrape (None for all available)

    Returns:
        List of article URLs
    """

    all_article_urls = []
    page_num = 1

    # Headers to mimic a real browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }

    print(f"�� Starting to scrape ChemXplore articles from: {base_url}")

    while True:
        # Construct the page URL
        if page_num == 1:
            page_url = base_url
        else:
            page_url = f"{base_url}?page={page_num}"

        print(f"\n📄 Scraping page {page_num}: {page_url}")

        try:
            # Make the request
            response = requests.get(page_url, headers=headers, timeout=30)
            response.raise_for_status()

            # Parse the HTML
            soup = BeautifulSoup(response.text, "html.parser")

            # Find ALL news item lists on the page (there are multiple!)
            news_lists = soup.find_all("ul", class_="news-item-list")

            if not news_lists:
                print(f"❌ No news lists found on page {page_num}")
                break

            print(f"✅ Found {len(news_lists)} news lists on page {page_num}")

            # Extract URLs from each news list
            page_article_urls = []
            total_items_found = 0

            for list_idx, news_list in enumerate(news_lists):
                # Find all news item previews in this list
                news_items = news_list.find_all("div", class_="news-item-preview")

                print(f"   �� List {list_idx + 1}: Found {len(news_items)} articles")

                # Extract URLs from each news item
                for item in news_items:
                    link = item.find("a", class_="news-item-preview__main")
                    if link and link.get("href"):
                        article_url = link["href"]
                        # Convert relative URLs to absolute URLs
                        if article_url.startswith("/"):
                            article_url = urljoin("https://chemxplore.com", article_url)

                        # Only add if not already in the list (avoid duplicates)
                        if article_url not in page_article_urls:
                            page_article_urls.append(article_url)
                            total_items_found += 1

            # Add to the main list
            all_article_urls.extend(page_article_urls)
            print(f"📝 Added {len(page_article_urls)} unique article URLs from page {page_num}")
            print(f"📊 Total articles found so far: {len(all_article_urls)}")

            # Check if we've reached the maximum pages
            if max_pages and page_num >= max_pages:
                print(f"✅ Reached maximum page limit ({max_pages})")
                break

            # Check if there are more pages using the "Next" link
            next_page_exists = check_next_page_exists(soup, page_num)

            if not next_page_exists:
                print(f"✅ No more pages found after page {page_num}")
                break

            page_num += 1

            # Add a small delay to be respectful to the server
            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"❌ Error scraping page {page_num}: {e}")
            break
        except Exception as e:
            print(f"❌ Unexpected error on page {page_num}: {e}")
            break

    print(f"\n�� Scraping completed!")
    print(f"📊 Total articles found: {len(all_article_urls)}")
    print(f"📄 Total pages scraped: {page_num}")

    return all_article_urls

def check_next_page_exists(soup, current_page):
    """
    Check if there's a next page available using the "Next" link
    """
    # Look for the "Next" link specifically
    next_link = soup.find("a", rel="next")
    if next_link:
        href = next_link.get("href")
        if href and "page=" in href:
            try:
                next_page_num = int(href.split("page=")[1])
                if next_page_num > current_page:
                    print(f"   ➡️ Found Next link to page {next_page_num}")
                    return True
            except ValueError:
                pass

    # Fallback: check if there are multiple news lists (indicates more content)
    news_lists = soup.find_all("ul", class_="news-item-list")
    if len(news_lists) > 1:
        # If we have multiple lists, there might be more pages
        return True

    return False

def save_urls_to_file(urls, filename="chemxplore_article_urls.txt"):
    """Save the URLs to a text file"""
    with open(filename, "w", encoding="utf-8") as f:
        for url in urls:
            f.write(url + "\n")
    print(f"💾 URLs saved to: {filename}")

def main():
    """Main function to run the scraper"""

    # Scrape all available pages
    article_urls = scrape_chemxplore_articles(max_pages=None)

    if article_urls:
        # Display first few URLs as examples
        print(f"\n📋 First 5 article URLs:")
        for i, url in enumerate(article_urls[:5], 1):
            print(f"{i}. {url}")

        if len(article_urls) > 5:
            print(f"... and {len(article_urls) - 5} more")

        # Save to file
        save_urls_to_file(article_urls)

        return article_urls
    else:
        print("❌ No articles found")
        return []

if __name__ == "__main__":
    # Run the scraper
    urls = main()

�� Starting to scrape ChemXplore articles from: https://chemxplore.com/news

📄 Scraping page 1: https://chemxplore.com/news
✅ Found 2 news lists on page 1
   �� List 1: Found 6 articles
   �� List 2: Found 22 articles
📝 Added 28 unique article URLs from page 1
📊 Total articles found so far: 28
   ➡️ Found Next link to page 2

📄 Scraping page 2: https://chemxplore.com/news?page=2
✅ Found 2 news lists on page 2
   �� List 1: Found 6 articles
   �� List 2: Found 22 articles
📝 Added 28 unique article URLs from page 2
📊 Total articles found so far: 56
   ➡️ Found Next link to page 3

📄 Scraping page 3: https://chemxplore.com/news?page=3
✅ Found 2 news lists on page 3
   �� List 1: Found 6 articles
   �� List 2: Found 22 articles
📝 Added 28 unique article URLs from page 3
📊 Total articles found so far: 84
   ➡️ Found Next link to page 4

📄 Scraping page 4: https://chemxplore.com/news?page=4
✅ Found 2 news lists on page 4
   �� List 1: Found 6 articles
   �� List 2: Found 22 articles
📝 Added 

KeyboardInterrupt: 

In [4]:
urls = scrape_chemxplore_articles(max_pages=3)

�� Starting to scrape ChemXplore articles from: https://chemxplore.com/news

📄 Scraping page 1: https://chemxplore.com/news
✅ Found 2 news lists on page 1
   �� List 1: Found 6 articles
   �� List 2: Found 22 articles
📝 Added 28 unique article URLs from page 1
📊 Total articles found so far: 28
   ➡️ Found Next link to page 2

📄 Scraping page 2: https://chemxplore.com/news?page=2
✅ Found 2 news lists on page 2
   �� List 1: Found 6 articles
   �� List 2: Found 22 articles
📝 Added 28 unique article URLs from page 2
📊 Total articles found so far: 56
   ➡️ Found Next link to page 3

📄 Scraping page 3: https://chemxplore.com/news?page=3
✅ Found 2 news lists on page 3
   �� List 1: Found 6 articles
   �� List 2: Found 22 articles
📝 Added 28 unique article URLs from page 3
📊 Total articles found so far: 84
✅ Reached maximum page limit (3)

�� Scraping completed!
📊 Total articles found: 84
📄 Total pages scraped: 3


In [2]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime

def extract_chemxplore_article(url):
    """
    Extract article content from a ChemXplore article
    with paragraphs structured as { p1, p2, ... }
    """

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Connection": "keep-alive",
    }

    try:
        # Fetch page
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # --- Title ---
        title_tag = soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else None

        # --- Date ---
        date_tag = soup.find("div", class_="article__date")
        date_str = date_tag.get_text(strip=True) if date_tag else None

        # Try to normalize to UTC ISO format
        date_utc = None
        if date_str:
            try:
                # Example: "Apr 4, 2025" → ISO UTC
                parsed = datetime.strptime(date_str, "%b %d, %Y")
                date_utc = parsed.strftime("%Y-%m-%dT00:00:00.000+00:00")
            except Exception:
                date_utc = date_str  # fallback to raw

        # --- Author ---
        author_tag = soup.find("div", class_="article__author")
        author = author_tag.get_text(strip=True) if author_tag else "Unknown"

        # --- Category ---
        category_tag = soup.find("div", class_="article__category")
        category = category_tag.get_text(strip=True) if category_tag else "Unknown"

        # --- Paragraphs ---
        raw_paragraphs = []
        article_content = soup.select_one("div.article__content") or soup.select_one("article")
        if article_content:
            for element in article_content.find_all(["p", "h2", "h3"]):
                text = element.get_text(strip=True)
                if text and len(text) > 5:
                    raw_paragraphs.append(text)

        if not raw_paragraphs:
            for p in soup.find_all("p"):
                text = p.get_text(strip=True)
                if text and len(text) > 10:
                    raw_paragraphs.append(text)

        # Convert paragraphs into p1, p2, ...
        paragraph_obj = {f"p{i+1}": text for i, text in enumerate(raw_paragraphs)}
        paragraphs = [paragraph_obj]  # wrap in array

        # --- Build Final Dict ---
        article_data = {
            "title": title,
            "paragraphs": paragraphs,
            "meta": {
                "date": date_utc,
                "url": url,
                "category": category,
                "author": author,
            },
        }

        return article_data

    except Exception as e:
        print(f"❌ Error: {e}")
        return None


# Example usage
if __name__ == "__main__":
    test_url = "https://chemxplore.com/news/lanxess-takes-action-to-counter-weak-market-environment"
    article_data = extract_chemxplore_article(test_url)

    if article_data:
        print(json.dumps(article_data, indent=2, ensure_ascii=False))
    else:
        print("❌ Failed to extract article data")


{
  "title": "LANXESS Responds to Market Challenges",
  "paragraphs": [
    {
      "p1": "Financial Performance",
      "p2": "In Q2 2025, sales decreased by 12.6% to EUR 1.47 billion, influenced by portfolio and volume effects. EBITDA pre exceptionals fell 17.1% year-on-year to EUR 150 million. Despite challenging conditions, the company achieved a positive free cash flow of EUR 31 million and reduced net financial debt by 18% to EUR 2.069 billion.",
      "p3": "Guidance Adjustment",
      "p4": "Due to ongoing weak demand, the company adjusted its fiscal year 2025 guidance, now expecting EBITDA pre exceptionals between EUR 520 million and EUR 580 million, down from the previous range of EUR 600 million to EUR 650 million. This revision includes a EUR 10 million impact from supply restrictions.",
      "p5": "Portfolio and Debt Management",
      "p6": "On April 1, 2025, the company sold its Urethane Systems business toUBE Corporation, marking a shift towards specialty chemicals. Pr